# Tableau Course Q&A Chatbot (Gemini + LangChain RAG)

### Environment Setup & API Configuration

In [1]:
%load_ext dotenv
%dotenv

### Import the Libraries : Document Loading & Metadata Enrichment

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter, TokenTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

### Load and Concatenate Course Transcript

In [6]:
loader_pdf = PyPDFLoader("Introduction_to_Tableau.pdf")

In [7]:
docs_list = loader_pdf.load()

In [8]:
string_list_concat = "".join([doc.page_content for doc in docs_list])

### Split Transcript by Markdown Headers (Section & Lecture Metadata)

In [9]:
md_splitter = MarkdownHeaderTextSplitter(
headers_to_split_on=[
("#", "Section Title"),
("##", "Lecture Title")
]
) 

In [10]:
docs_list_md_split = md_splitter.split_text(string_list_concat)

### LLM-Based Transcript Formatting & Cleanup Chain

In [11]:
string_list_split = [i.page_content for i in docs_list_md_split]

In [12]:
PROMPT_FORMATTING_S = '''Improve the following Tableau lecture transcript by:
- Splitting the text into meaningful paragraphs
- Correcting any misplaced punctuation
- Fixing mistranscribed words (e.g., changing 'tableaux' to 'Tableau')"
'''

PROMPT_TEMPLATE_FORMATTING_H = '''This is the transcript:
{lecture_transcript}
'''

In [14]:
prompt_formatting_s = SystemMessage(PROMPT_FORMATTING_S)
prompt_template_formatting_h = HumanMessagePromptTemplate.from_template(PROMPT_TEMPLATE_FORMATTING_H)
chat_prompt_template_formatting = ChatPromptTemplate([prompt_formatting_s, prompt_template_formatting_h])

In [ ]:
chat = ChatGoogleGenerativeAI(
model="gemini-3.5-flash-lite",
seed=365
)

In [16]:
str_output_parser = StrOutputParser()

In [17]:
chain_formatting = chat_prompt_template_formatting | chat | str_output_parser

In [20]:
string_list_formatted = chain_formatting.batch([{"lecture_transcript": text} for text in string_list_split])

In [67]:
# Override the docs_list_md_split list such that the page_content parameter of each Document objects stores the updated lecture.
[setattr(i, "page_content", j) for i, j in zip(docs_list_md_split, string_list_formatted)];

In [21]:
print(string_list_formatted[0])

Here is the improved transcript:

Hi everyone. I'm Ned, and I'll be your instructor for this course. 

Tableau is an invaluable tool one needs to learn on their journey to become a successful business intelligence analyst or data scientist. 

The art of these professions is storytelling—using data to tell stories and convince top management of the right course of action. 

By completing this part of the program, you will know how to create charts and dashboards in Tableau. This is an essential step on your way to a data scientist role.


### Token-Based Document Chunking

In [28]:
%pip install tiktoken

   ---------------------------------------- 0.0/941.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/941.2 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/941.2 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/941.2 kB ? eta -:--:--
   --------------------- ---------------- 524.3/941.2 kB 799.2 kB/s eta 0:00:01
   ---------------------------------------- 941.2/941.2 kB 988.4 kB/s  0:00:01

   -------------------- ------------------- 1/2 [tiktoken]
   ---------------------------------------- 2/2 [tiktoken]

Note: you may need to restart the kernel to use updated packages.


In [29]:
token_splitter = TokenTextSplitter(encoding_name="cl100k_base", chunk_size=500, chunk_overlap=50)

In [30]:
docs_list_tokens_split = token_splitter.split_documents(docs_list_md_split)

### Embedding Generation, Chroma Vector Store & MMR Retriever

In [41]:
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [42]:
vectorstore = Chroma.from_documents(
documents=docs_list_tokens_split,
embedding=embedding,
persist_directory="./intro-to-tableau"
)

In [43]:
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 2, "lambda_mult": 0.7})

### Prompt Engineering & Chat Prompt Templates

In [46]:
PROMPT_CREATING_QUESTION = '''Lecture: {question_lecture}
Title: {question_title}
Body: {question_body}'''

PROMPT_RETRIEVING_S = '''You will receive a question from a student taking a Tableau course, which includes a title and a body. 
The corresponding lecture will also be provided.

Answer the question using only the provided context.

At the end of your response, include the section and lecture names where the context was drawn from, formatted as follows: 
Resources: 
Section: *Section Title*, Lecture: *Lecture Title* 
...
Replace *Section Title* and *Lecture Title* with the appropriate titles.'''

PROMPT_TEMPLATE_RETRIEVING_H = '''This is the question:
{question}

This is the context:
{context}'''

from langchain_core.prompts import PromptTemplate
prompt_creating_question = PromptTemplate.from_template(PROMPT_CREATING_QUESTION)
prompt_retrieving_s = SystemMessage(PROMPT_RETRIEVING_S)
prompt_template_retrieving_h = HumanMessagePromptTemplate.from_template(PROMPT_TEMPLATE_RETRIEVING_H)

chat_prompt_template_retrieving = ChatPromptTemplate([prompt_retrieving_s, prompt_template_retrieving_h])

### Initial Q&A Retrieval Chain (`chain_retrieving`)

In [49]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [50]:
chain_retrieving = (
prompt_creating_question
| RunnableLambda(lambda x: x.text)
| {"context": retriever, "question": RunnablePassthrough()}
| chat_prompt_template_retrieving
| chat
| str_output_parser
)

In [51]:
result = chain_retrieving.invoke({"question_lecture": "Adding a custom calculation",
                                  "question_title": "Why are we using SUM here? It's unclear to me.",
                                  "question_body": "This question refers to calculating the GM%."})

In [52]:
result

'We use `SUM` around the variables when calculating GM% because, without it, we are not dividing the total figures for each month (the aggregated totals). If you do not sum the variables, Tableau does not perform the calculation on the aggregated total figures for the period, which can result in incorrect values (such as numbers way outside the expected percentage range). \n\nResources: \nSection: Introduction to Tableau, Lecture: Creating a custom field and adding calculations to a table'

### Improved Q&A Retrieval Chain (chain_retrieving_improved)

In [56]:
from langchain_core.runnables import chain

@chain
def format_context(dictionary):
 return {"context": "\n\n".join([f"Document {i+1}\nSection Title: {doc.metadata.get('Section Title', '')}\nLecture Title: {doc.metadata.get('Lecture Title', '')}\nContent: {doc.page_content}\n\n-------------------" for i, doc in enumerate(dictionary["context"])]), "question": dictionary["question"]}

In [58]:
chain_retrieving_improved = (
prompt_creating_question
| RunnableLambda(lambda x: x.text)
| {"context": retriever, "question": RunnablePassthrough()}
| format_context
| chat_prompt_template_retrieving
| chat
| str_output_parser
)

In [59]:
result_improved = chain_retrieving_improved.invoke({"question_lecture": "Adding a custom calculation",
                                                    "question_title": "Why are we using SUM here? It's unclear to me.",
                                                    "question_body": "This question refers to calculating the GM%."})

In [60]:
result_improved

'Based on the provided context, `SUM` is used in the calculation because without it, you are not dividing the total figures for each variable (gross profit and revenue) for each month. \n\nResources: \nSection: Introduction to Tableau, Lecture: Adding a custom calculation'

### Stream Response Evaluation

In [66]:
result_streamed = chain_retrieving_improved.stream({"question_lecture": "Adding a custom calculation","question_title": "Why are we using SUM here? It's unclear to me.","question_body": "This question refers to calculating the GM%."
})
for chunk in result_streamed:
 print(chunk, end="")

Based on the provided context, `SUM` is used in the calculation because without it, you are not dividing the total figures for each variable (gross profit and revenue) for each month. 

Resources: 
Section: Introduction to Tableau, Lecture: Adding a custom calculation

In [ ]:
# Create a for-loop to stream the response
